# 04 — Sequence-level VAE (model M2)

Each 2 s window (60 frames × 134 features) is compressed into **one latent vector z** (64 numbers), then decoded back into the whole window.
**The model never sees speed labels.** Speed is used afterwards only, to test whether it ended up as a direction in z.

```
window (60×134) → 1D-conv encoder → μ, log σ² (64) → z → 1D-conv decoder → window (60×134)
```

**Loss** = reconstruction (squared error; foot contacts as yes/no) + β · KL divergence. β is warmed up from 0 over the first epochs.

**The same notebook trains three variants** (settings cell):
| Setting | Model |
|---|---|
| `BETA = 1.0`, `USE_NOROOT = False` | **VAE** (main model) |
| `BETA = 0`, `USE_NOROOT = False` | **plain autoencoder** (no randomness, no KL) |
| `USE_NOROOT = True` | **root-ablation model (M3)**, trained on the features without root speed |

**After training, the notebook checks:**
1. Loss curves
2. Reconstruction error on **test people** (features and joint positions in cm)
3. Whether walking speed survives reconstruction
4. **Linear probe:** ridge regression z → speed on test people. If speed can't be read linearly from z, there's no linear direction to find.
5. Latent space coloured by speed, reconstructions and random samples as stick figures
6. A first **preview of the speed direction**: move z along the probe direction and measure the decoded speed

Everything is saved to `models/<run name>/` next to the `preprocessed` folder.

## Settings

In [ ]:
from pathlib import Path

PRE_DIR = Path(r"C:\Users\hasna\Documents\Good Practise\CMU\walking\preprocessed")   # output of notebook 02

USE_NOROOT  = False   # True = root-ablation model (trained on X_noroot)
BETA        = 1.0     # KL weight; 0 = plain autoencoder
LATENT_DIM  = 64
HIDDEN      = 256     # channels in the conv layers
ROOT_WEIGHT = 2.0     # extra loss weight on root height / speed / turning channels (full features only)

EPOCHS           = 300
BATCH_SIZE       = 128
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
KL_WARMUP_EPOCHS = 30     # β grows from 0 to BETA over these epochs
PATIENCE         = 30     # stop if the val loss hasn't improved for this many epochs
SEED             = 0

RUN_NAME = f"{'ae' if BETA == 0 else 'vae'}_{'noroot' if USE_NOROOT else 'full'}_z{LATENT_DIM}_b{BETA:g}_s{SEED}"
OUT_DIR  = PRE_DIR.parent / "models" / RUN_NAME

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore", message=".*[Ii]ll-conditioned.*")

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| device:", device, "| run:", RUN_NAME)

cfg  = json.loads((PRE_DIR / "config.json").read_text())
norm = np.load(PRE_DIR / "normalization.npz")
FPS  = cfg["TARGET_FPS"]
BONES = cfg["BONES"]
FEATS = cfg["FEATURES_NOROOT"] if USE_NOROOT else cfg["FEATURES"]
MEAN  = norm["mean_noroot"] if USE_NOROOT else norm["mean"]
STD   = norm["std_noroot"] if USE_NOROOT else norm["std"]
CONTACT = [i for i, f in enumerate(FEATS) if f.startswith("contact_")]
REGULAR = [i for i in range(len(FEATS)) if i not in CONTACT]
ROOT    = [i for i, f in enumerate(FEATS) if f.startswith("root_")]
FWD     = FEATS.index("root_vel_forward") if "root_vel_forward" in FEATS else None

data = {}
for s in ["train", "val", "test"]:
    d = np.load(PRE_DIR / f"{s}.npz", allow_pickle=True)
    m = pd.read_csv(PRE_DIR / f"{s}_meta.csv")
    data[s] = dict(X=d["X_noroot" if USE_NOROOT else "X"].astype(np.float32), joints=d["joints"], meta=m)
    print(f"{s:5s}: {data[s]['X'].shape}  people={m.person.nunique()}")
T, D = data["train"]["X"].shape[1:]
assert T % 4 == 0, f"window length {T} must be divisible by 4 (change WINDOW_FRAMES in notebook 02)"

## Model

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, k=5):
        super().__init__()
        self.net = nn.Sequential(nn.GroupNorm(8, c), nn.GELU(), nn.Conv1d(c, c, k, padding=k // 2),
                                 nn.GroupNorm(8, c), nn.GELU(), nn.Conv1d(c, c, k, padding=k // 2))
    def forward(self, x):
        return x + self.net(x)

class MotionVAE(nn.Module):
    def __init__(self, D, T, latent=LATENT_DIM, H=HIDDEN):
        super().__init__()
        self.T4, self.H = T // 4, H
        self.enc = nn.Sequential(
            nn.Conv1d(D, H, 5, padding=2), ResBlock(H),
            nn.Conv1d(H, H, 4, stride=2, padding=1), ResBlock(H),      # T/2
            nn.Conv1d(H, H, 4, stride=2, padding=1), ResBlock(H),      # T/4
            nn.Flatten(), nn.Linear(H * self.T4, 2 * latent))
        self.dec_in = nn.Linear(latent, H * self.T4)
        self.dec = nn.Sequential(
            ResBlock(H), nn.ConvTranspose1d(H, H, 4, stride=2, padding=1),   # T/2
            ResBlock(H), nn.ConvTranspose1d(H, H, 4, stride=2, padding=1),   # T
            ResBlock(H), nn.GroupNorm(8, H), nn.GELU(), nn.Conv1d(H, D, 5, padding=2))

    def encode(self, x):                       # x: (B, T, D)
        mu, logvar = self.enc(x.transpose(1, 2)).chunk(2, dim=-1)
        return mu, logvar.clamp(-10, 10)

    def decode(self, z):                       # -> (B, T, D); contact channels are logits
        h = self.dec_in(z).view(-1, self.H, self.T4)
        return self.dec(h).transpose(1, 2)

    def forward(self, x, sample=True):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * (0.5 * logvar).exp() if (sample and BETA > 0) else mu
        return self.decode(z), mu, logvar

model = MotionVAE(D, T).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

w = torch.ones(D); w[ROOT] = ROOT_WEIGHT
W_REG = w[REGULAR].to(device)
def loss_terms(xh, x, mu, logvar):
    rec = ((xh[..., REGULAR] - x[..., REGULAR]) ** 2 * W_REG).sum((1, 2))
    if CONTACT:
        rec = rec + Fnn.binary_cross_entropy_with_logits(xh[..., CONTACT], x[..., CONTACT], reduction="none").sum((1, 2))
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(-1)
    return rec.mean(), kl.mean()

## Train
Early stopping keeps the epoch with the lowest val loss (reconstruction + full β·KL).

In [ ]:
Xtr = torch.tensor(data["train"]["X"]); Xva = torch.tensor(data["val"]["X"]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=PATIENCE // 3)

hist, best, best_state, wait = [], np.inf, None, 0
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    beta_t = BETA * min(1.0, epoch / max(1, KL_WARMUP_EPOCHS))
    model.train()
    perm = torch.randperm(len(Xtr))
    tr_rec = tr_kl = 0.0
    for i in range(0, len(Xtr), BATCH_SIZE):
        xb = Xtr[perm[i:i + BATCH_SIZE]].to(device)
        xh, mu, logvar = model(xb)
        rec, kl = loss_terms(xh, xb, mu, logvar)
        loss = rec + beta_t * kl
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_rec += rec.item() * len(xb); tr_kl += kl.item() * len(xb)
    model.eval()
    with torch.no_grad():
        xh, mu, logvar = model(Xva, sample=False)
        va_rec, va_kl = (v.item() for v in loss_terms(xh, Xva, mu, logvar))
    va_loss = va_rec + BETA * va_kl
    sched.step(va_loss)
    hist.append(dict(epoch=epoch, beta=beta_t, train_rec=tr_rec / len(Xtr), train_kl=tr_kl / len(Xtr),
                     val_rec=va_rec, val_kl=va_kl, val_loss=va_loss, lr=opt.param_groups[0]["lr"]))
    if va_loss < best - 1e-4 and epoch >= min(KL_WARMUP_EPOCHS, EPOCHS):
        best, best_state, wait = va_loss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, 0
    elif epoch >= KL_WARMUP_EPOCHS:
        wait += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:4d} | train rec {tr_rec/len(Xtr):9.2f} kl {tr_kl/len(Xtr):7.2f} | "
              f"val rec {va_rec:9.2f} kl {va_kl:7.2f} | β {beta_t:.2f} | {time.time()-t0:5.0f}s")
    if wait >= PATIENCE:
        print(f"early stop at epoch {epoch}")
        break

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state); model.eval()
hist = pd.DataFrame(hist); hist.to_csv(OUT_DIR / "history.csv", index=False)
torch.save(dict(state_dict=best_state, D=int(D), T=int(T), LATENT_DIM=LATENT_DIM, HIDDEN=HIDDEN, BETA=BETA,
                USE_NOROOT=USE_NOROOT, features=FEATS, pre_dir=str(PRE_DIR)), OUT_DIR / "model.pt")
print("saved", OUT_DIR / "model.pt")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(hist.epoch, hist.train_rec, label="train"); ax[0].plot(hist.epoch, hist.val_rec, label="val")
ax[0].set(yscale="log", xlabel="epoch", title="reconstruction loss"); ax[0].legend()
ax[1].plot(hist.epoch, hist.train_kl, label="train"); ax[1].plot(hist.epoch, hist.val_kl, label="val")
ax[1].set(xlabel="epoch", title="KL divergence"); ax[1].legend()
plt.tight_layout(); plt.show()

## Encode every window and decode it back

In [ ]:
def rot_xy(v, ang):
    c, s = np.cos(ang), np.sin(ang)
    shape = (len(ang),) + (1,) * (v.ndim - 2)
    c, s = c.reshape(shape), s.reshape(shape)
    out = v.copy()
    out[..., 0] = c * v[..., 0] - s * v[..., 1]
    out[..., 1] = s * v[..., 0] + c * v[..., 1]
    return out

def recover_joints(x, fps=FPS):
    # full features (T, 134), de-normalised -> joints (T, 22, 3)  [same as notebook 02]
    T_ = len(x)
    th = np.r_[0.0, np.cumsum(x[:-1, 3]) / fps]
    v = rot_xy(np.c_[x[:, 1:3], np.zeros(T_)], th)[:, :2]
    root_xy = np.r_[np.zeros((1, 2)), np.cumsum(v[:-1], 0) / fps]
    local = rot_xy(x[:, 4:67].reshape(T_, 21, 3).astype(np.float64), th)
    local[..., :2] += root_xy[:, None]
    return np.concatenate([np.c_[root_xy, x[:, 0]][:, None], local], 1)

def local_pose(x):
    # root-centred pose in the heading frame (T, 22, 3); works for both feature layouts
    T_ = len(x)
    pos = x[:, 1:64] if USE_NOROOT else x[:, 4:67]
    root = np.zeros((T_, 1, 3)); root[:, 0, 2] = x[:, 0]
    return np.concatenate([root, pos.reshape(T_, 21, 3)], 1)

def denorm(xn):
    x = xn * STD + MEAN
    if CONTACT:
        x[..., CONTACT] = 1 / (1 + np.exp(-xn[..., CONTACT]))   # logits -> probability
    return x

@torch.no_grad()
def encode(X, bs=1024):
    out = [model.encode(torch.tensor(X[i:i + bs]).to(device)) for i in range(0, len(X), bs)]
    return torch.cat([o[0] for o in out]).cpu().numpy(), torch.cat([o[1] for o in out]).cpu().numpy()

@torch.no_grad()
def decode(Z, bs=1024):
    return np.concatenate([model.decode(torch.tensor(Z[i:i + bs], dtype=torch.float32).to(device)).cpu().numpy()
                           for i in range(0, len(Z), bs)])

for s in data:
    data[s]["mu"], data[s]["logvar"] = encode(data[s]["X"])
    data[s]["recon"] = denorm(decode(data[s]["mu"]))
    data[s]["orig"] = data[s]["X"] * STD + MEAN
    np.savez(OUT_DIR / f"latents_{s}.npz", mu=data[s]["mu"], logvar=data[s]["logvar"],
             window_id=data[s]["meta"].window_id.values, speed=data[s]["meta"].fwd_speed_mps.values)
print("latents saved")

## Reconstruction quality on test people
- **Feature RMSE**: in normalised units (1 = one standard deviation).
- **Pose error**: mean joint distance with the root fixed, in cm.
- **Full error**: mean joint distance including where the body went (full model only). This includes drift from integrating the root speed.

In [ ]:
te = data["test"]
xn_rec = (te["recon"] - MEAN) / STD
feat_rmse = float(np.sqrt(((xn_rec[..., REGULAR] - te["X"][..., REGULAR]) ** 2).mean()))
pose_err = float(np.mean([np.linalg.norm(local_pose(r) - local_pose(o), axis=-1).mean()
                          for r, o in zip(te["recon"], te["orig"])]) * 100)
res = dict(run=RUN_NAME, test_feature_rmse=feat_rmse, test_pose_error_cm=pose_err)
if CONTACT:
    res["test_contact_accuracy"] = float(((te["recon"][..., CONTACT] > 0.5) == (te["orig"][..., CONTACT] > 0.5)).mean())
if not USE_NOROOT:
    rj = np.stack([recover_joints(r) for r in te["recon"]])
    res["test_full_error_cm"] = float(np.linalg.norm(rj - te["joints"], axis=-1).mean() * 100)
    res["test_end_drift_cm"] = float(np.linalg.norm(rj[:, -1, 0, :2] - te["joints"][:, -1, 0, :2], axis=-1).mean() * 100)
    rec_speed = te["recon"][..., FWD].mean(1)
    true_speed = te["meta"].fwd_speed_mps.values
    res.update(speed_after_recon_R2=float(r2_score(true_speed, rec_speed)),
               speed_after_recon_MAE=float(mean_absolute_error(true_speed, rec_speed)))
for k, v in res.items():
    print(f"{k:26s} {v}")

## Linear probe: can speed be read from z? (test people)
Ridge regression from μ to forward speed, fit on train, strength chosen on val. **High test R² means speed is (close to) a linear direction in z.** This is the gate before Phase 4.

In [ ]:
y = {s: data[s]["meta"].fwd_speed_mps.values for s in data}
bestp = None
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    p = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(data["train"]["mu"], y["train"])
    v = r2_score(y["val"], p.predict(data["val"]["mu"]))
    if bestp is None or v > bestp[0]:
        bestp = (v, p, a)
_, probe, alpha = bestp
pred = probe.predict(data["test"]["mu"])
res.update(probe_alpha=alpha, probe_val_R2=float(bestp[0]),
           probe_test_R2=float(r2_score(y["test"], pred)),
           probe_test_MAE=float(mean_absolute_error(y["test"], pred)),
           probe_test_spearman=float(spearmanr(y["test"], pred)[0]))
print({k: round(v, 3) for k, v in res.items() if k.startswith("probe")})

# speed direction in z units: d = probe weight / scaler scale, unit length
d_speed = probe[-1].coef_ / probe[0].scale_
d_speed /= np.linalg.norm(d_speed)
np.save(OUT_DIR / "speed_direction_probe.npy", d_speed)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
lo, hi = y["test"].min() - .1, y["test"].max() + .1
ax[0].scatter(y["test"], pred, s=10, alpha=.6); ax[0].plot([lo, hi], [lo, hi], "k--", lw=1)
ax[0].set(xlabel="true speed (m/s)", ylabel="predicted from z", title=f"probe on test people (R²={res['probe_test_R2']:.2f})")
pca = PCA(2).fit(data["train"]["mu"])
for s, mk in [("train", "."), ("test", "x")]:
    P = pca.transform(data[s]["mu"])
    sc = ax[1].scatter(P[:, 0], P[:, 1], c=y[s], cmap="viridis", s=12, marker=mk, alpha=.7, label=s)
plt.colorbar(sc, ax=ax[1], label="speed (m/s)"); ax[1].legend(); ax[1].set(title="latent space (PCA), colour = speed")
proj = {s: data[s]["mu"] @ d_speed for s in data}
ax[2].scatter(proj["train"], y["train"], s=6, alpha=.4, label="train"); ax[2].scatter(proj["test"], y["test"], s=10, alpha=.7, label="test")
ax[2].set(xlabel="z projected on speed direction", ylabel="speed (m/s)", title="speed along the probe direction"); ax[2].legend()
plt.tight_layout(); plt.show()

## Stick figures: original vs. reconstruction, and random samples
Side view (x forward, z up), 6 frames per window. Original in grey, reconstruction in colour.

In [ ]:
def draw(ax, P, color, alpha=1.0, lw=1.5):
    for j, p in BONES:
        ax.plot([P[j, 0], P[p, 0]], [P[j, 2], P[p, 2]], color=color, alpha=alpha, lw=lw)

def skeleton(x):
    return local_pose(x) if USE_NOROOT else recover_joints(x)

def show(ax, x, ref=None, title="", spread=0.6):
    for k, t in enumerate(np.linspace(0, len(x) - 1, 6).astype(int)):
        off = np.array([k * spread, 0, 0]) if USE_NOROOT else 0
        if ref is not None:
            draw(ax, skeleton(ref)[t] + off, "lightgrey", lw=3)
        draw(ax, skeleton(x)[t] + off, plt.cm.viridis(k / 5))
    ax.axhline(0, color="grey", lw=.6); ax.set_aspect("equal"); ax.set_title(title, fontsize=9)

rng = np.random.default_rng(SEED)
idx = rng.choice(len(te["recon"]), size=min(2, len(te["recon"])), replace=False)
# random samples: N(0, I) for a VAE; for a plain AE, a Gaussian fitted to the train latents
if BETA > 0:
    Zs = rng.standard_normal((2, LATENT_DIM))
else:
    Zs = rng.multivariate_normal(data["train"]["mu"].mean(0), np.cov(data["train"]["mu"].T), 2)
samples = denorm(decode(Zs.astype(np.float32)))

fig, axes = plt.subplots(4, 1, figsize=(12, 11))
for ax, i in zip(axes[:2], idx):
    m = te["meta"].iloc[i]
    show(ax, te["recon"][i], te["orig"][i], f"test reconstruction | {m.window_id} | {m.description} | true {m.fwd_speed_mps:.2f} m/s")
for ax, s in zip(axes[2:], samples):
    sp = f" | decoded speed {s[:, FWD].mean():.2f} m/s" if FWD is not None else ""
    show(ax, s, None, "random sample from the latent space" + sp)
plt.tight_layout(); plt.show()

## Preview: edit z along the speed direction
z′ = z + α · σ · d, where d is the probe direction and σ is the spread of train windows along it.
Test windows are decoded at several α and their **measured forward speed** is compared. A rising curve means moving along d changes the speed of the decoded walk.
This is only a first look; Phase 4/5 does it properly (several direction methods, plausibility, cadence vs. stride).

In [ ]:
if FWD is None:
    print("No root-speed channel in the no-root model, so speed can't be measured from its output. "
          "Use the probe result above for this model.")
else:
    sigma = proj["train"].std()
    alphas = np.linspace(-3, 3, 13)
    base = data["test"]["mu"]
    speeds = np.stack([denorm(decode((base + a * sigma * d_speed).astype(np.float32)))[..., FWD].mean(1) for a in alphas], 1)
    rho = np.array([spearmanr(alphas, s)[0] for s in speeds])
    res.update(edit_mean_spearman=float(np.nanmean(rho)),
               edit_speed_at_minus3=float(speeds[:, 0].mean()), edit_speed_at_0=float(speeds[:, 6].mean()),
               edit_speed_at_plus3=float(speeds[:, -1].mean()))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(alphas, speeds.T, color="tab:blue", alpha=.08)
    ax[0].plot(alphas, speeds.mean(0), color="k", lw=2, label="mean over test windows")
    ax[0].set(xlabel="α (σ units along the speed direction)", ylabel="decoded forward speed (m/s)", title="speed edit preview"); ax[0].legend()
    ax[1].hist(rho, bins=20); ax[1].set(xlabel="Spearman ρ (α vs. speed) per window", title=f"monotonicity (mean ρ = {np.nanmean(rho):.2f})")
    plt.tight_layout(); plt.show()
    ex = speeds.shape[0] // 2
    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    for ax, a in zip(axes, [-2.0, 0.0, 2.0]):
        x = denorm(decode((base[ex:ex + 1] + a * sigma * d_speed).astype(np.float32)))[0]
        show(ax, x, None, f"α = {a:+.0f}  →  decoded speed {x[:, FWD].mean():.2f} m/s")
    plt.tight_layout(); plt.show()

## Save the results

In [ ]:
res.update(epochs_trained=int(hist.epoch.max()), best_val_loss=float(best), n_train=len(data["train"]["X"]),
           settings=dict(BETA=BETA, LATENT_DIM=LATENT_DIM, HIDDEN=HIDDEN, USE_NOROOT=USE_NOROOT, ROOT_WEIGHT=ROOT_WEIGHT,
                         LR=LR, BATCH_SIZE=BATCH_SIZE, SEED=SEED))
(OUT_DIR / "results.json").write_text(json.dumps(res, indent=2))
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items() if k != "settings"}, indent=2))
print("\nfiles in", OUT_DIR, ":", sorted(p.name for p in OUT_DIR.iterdir()))

## What to look for
- **Pose error** a few cm and **speed after reconstruction R²** above ~0.9: the VAE reconstructs walking well.
- **Probe test R²**: compare it with notebook 03's no-root ridge result. Similar or higher means the latent kept the speed information linearly. Much lower means the VAE is squeezing it out. Try a lower `BETA` (e.g. 0.1) or a larger `LATENT_DIM`.
- **KL near 0** for many epochs means *posterior collapse* (z is ignored). Lower `BETA`.
- **Random samples** should look like walking. If they don't but reconstructions do, `BETA` is probably too low for good sampling (not a problem for the edit experiments).
- **Next runs** for the comparison: `BETA = 0` (plain AE), `USE_NOROOT = True` (M3), and 2 more `SEED`s for each. Every run gets its own folder.